In [1]:
import os
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# Cấu hình

In [ ]:
DATA = Path(os.environ.get("TIKI_DATA", "tiki_du_lieu_nghien_cuu.xlsx"))
SHEET = "Du_lieu_nghien_cuu"
OUT = Path(os.environ.get("TIKI_OUT", "eda_donbien"))
OUT.mkdir(parents=True, exist_ok=True)

NUM_COLS = ["quantity_sold", "sold_log", "sold_per_day",
            "log_price_ratio", "price", "n_images", "desc_len", "n_specs",
            "s_avg_rating_point", "review_count", "rating_average", "s_total_follower"]
BINARY_COLS = ["has_video", "s_is_official", "is_tiki_trading", "is_branded", "in_velocity"]
CAT_COLS = ["main_category"]

# Biến hình thành sau khi bán 
CONSEQ_SET = {"review_count", "rating_average", "s_total_follower"}


ZERO_AS_MISSING = {"rating_average": "review_count"}
IQR_K = 1.5          
SKEW_LIMIT = 1.0     
BINS = 40
MIN_N_PANEL = 30

sns.set_theme(style="whitegrid", font_scale=0.9)
PDF = PdfPages(OUT / "eda_donbien_tong_hop.pdf")

# Hàm hỗ trợ

In [3]:
def save(fig, name):
    path = OUT / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    PDF.savefig(fig, bbox_inches="tight")
    plt.close(fig)
    print(f"  -> đã lưu {path.name}")


def label(c):
    tag = " (hệ quả)" if c in CONSEQ_SET else ""
    return c + tag


def panel_grid(n, ncol=4, w=3.6, h=3.1):
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(w * ncol, h * nrow), squeeze=False)
    for k in range(n, nrow * ncol):
        axes.flat[k].axis("off")
    return fig, axes.flat


def iqr_outliers(s):
    s = s.dropna()
    q1, q3 = s.quantile([.25, .75])
    i = q3 - q1
    return int(((s < q1 - IQR_K * i) | (s > q3 + IQR_K * i)).sum())


def mode_info(s):
    vc = s.dropna().value_counts()
    return vc.index[0], vc.iloc[0] / vc.sum()

# Đọc dữ liệu

In [ ]:
print(f"[Đọc] {DATA} (sheet {SHEET})")
df = pd.read_excel(DATA, sheet_name=SHEET)

required = NUM_COLS + BINARY_COLS + CAT_COLS
missing_cols = sorted(set(c for c in required if c not in df.columns))
if missing_cols:
    raise SystemExit("File thiếu các cột: " + ", ".join(missing_cols))

num_cols = list(NUM_COLS)
bin_cols = [c for c in BINARY_COLS if c in df.columns]
cat_cols = [c for c in CAT_COLS if c in df.columns]

print(f"[Mẫu] {len(df):,} sản phẩm × {df.shape[1]} cột | trùng product_id: {df['product_id'].duplicated().sum()}")
print(f"[Biến] định lượng {len(num_cols)} | nhị phân {len(bin_cols)} | phân loại {len(cat_cols)}")

[Đọc] tiki_du_lieu_nghien_cuu.xlsx (sheet Du_lieu_nghien_cuu)
[Mẫu] 2,034 sản phẩm × 22 cột | trùng product_id: 0
[Biến] định lượng 12 | nhị phân 5 | phân loại 1


# Bước 1: Tổng quan dữ liệu và Phân tích đơn biến

## 1a: Thống kê mô tả

In [ ]:
def fmt(x)
    if pd.isna(x):
        return ""
    return f"{int(x)}" if float(x).is_integer() else f"{x:.3f}"

rows = []
for c in num_cols:
    s = df[c].dropna()
    mv, ms = mode_info(s)
    n_out = iqr_outliers(s)
    rows.append({
        "bien": label(c), "count": len(s), "n_thieu": int(df[c].isna().sum()),
        "n_bang_0": int((s == 0).sum()),
        "TB": s.mean(), "TV": s.median(), "SD": s.std(),
        "min": s.min(), "Q1": s.quantile(.25), "Q3": s.quantile(.75), "max": s.max(),
    })
desc = pd.DataFrame(rows).set_index("bien")
pd.set_option("display.width", 250, "display.max_columns", 30)
desc_show = desc.astype(float).apply(lambda col: col.map(fmt))
print(desc_show.to_string())
desc.round(3).to_csv(OUT / "bang_1a_thong_ke_mo_ta.csv", encoding="utf-8-sig")


=== Bước 1a. Thống kê mô tả biến định lượng ===
                              count n_thieu n_bang_0          TB        TV           SD     min        Q1       Q3       max
bien                                                                                                                        
quantity_sold                  2034       0        0    1159.285        50     4950.340       1         9  337.750     92827
sold_log                       2034       0        0       4.199     3.932        2.365   0.693     2.303    5.825    11.439
sold_per_day                   1996      38        0       1.446     0.051       19.916   0.000     0.012    0.303       844
log_price_ratio                2034       0        8      -0.197    -0.253        1.137  -4.396    -0.889    0.365     5.307
price                          2034       0        0  886402.543    220000  3796313.658    3750    117900   499000  69030000
n_images                       2034       0        0       6.090         5  

## 1b: Phân phối từng biến

In [ ]:
fig, axs = panel_grid(len(num_cols))
for ax, c in zip(axs, num_cols):
    s = df[c].dropna()
    sns.histplot(s, bins=BINS, kde=s.nunique() > 10, color="#4C72B0", ax=ax)
    t = f"{label(c)}\nn={len(s):,}, skew={s.skew():.2f}"
    if len(s) < MIN_N_PANEL:
        t += " (ít dữ liệu)"
    ax.set_title(t, fontsize=8.5)
    ax.set_xlabel("")
    ax.set_ylabel("số sản phẩm" if ax.get_subplotspec().is_first_col() else "")
fig.suptitle("Bước 1b-1. Histogram", y=1.01)
fig.tight_layout()
save(fig, "1b1_histogram")

fig, axs = panel_grid(len(num_cols))
for ax, c in zip(axs, num_cols):
    s = df[c].dropna()
    sns.boxplot(x=s, color="#9ecae1", fliersize=2, ax=ax)
    ax.set_title(f"{label(c)}\nngoại lai IQR: {iqr_outliers(s):,}", fontsize=8.5)
    ax.set_xlabel("")
fig.suptitle("Bước 1b-2. Boxplot", y=1.01)
fig.tight_layout()
save(fig, "1b2_boxplot")


=== Bước 1b. Phân phối từng biến ===
  -> đã lưu 1b1_histogram.png
  -> đã lưu 1b2_boxplot.png


## 1c: Biến lệch phải: gốc và log1p

In [ ]:
skewed = [c for c in num_cols if df[c].skew() > SKEW_LIMIT and df[c].min() >= 0]
print("Biến lệch phải (skew >", SKEW_LIMIT, "):", skewed)
fig, axs = panel_grid(2 * len(skewed))
rows2c = []
for k, c in enumerate(skewed):
    s = df[c].dropna()
    ls = np.log1p(s)
    sns.histplot(s, bins=BINS, color="#4C72B0", ax=axs[2 * k])
    axs[2 * k].set_title(f"{label(c)} (gốc)\nskew={s.skew():.2f}", fontsize=8.5)
    sns.histplot(ls, bins=BINS, kde=True, color="#55A868", ax=axs[2 * k + 1])
    axs[2 * k + 1].set_title(f"log1p({c})\nskew={ls.skew():.2f}", fontsize=8.5)
    for a in (axs[2 * k], axs[2 * k + 1]):
        a.set_xlabel(""); a.set_ylabel("")
    rows2c.append({"bien": c, "skew_goc": s.skew(), "skew_log1p": ls.skew()})
fig.suptitle("Bước 1c. Trước và sau log1p", y=1.01)
fig.tight_layout()
save(fig, "1c_goc_vs_log")
tab2c = pd.DataFrame(rows2c).set_index("bien")
print(tab2c.round(2).to_string())
tab2c.to_csv(OUT / "bang_1c_skew_truoc_sau_log.csv", encoding="utf-8-sig")


=== Bước 1c. Biến lệch phải: gốc và log1p ===
Biến lệch phải (skew > 1.0 ): ['quantity_sold', 'sold_per_day', 'price', 'n_images', 'desc_len', 'n_specs', 'review_count']
  -> đã lưu 1c_goc_vs_log.png
               skew_goc  skew_log1p
bien                               
quantity_sold      9.71        0.49
sold_per_day      38.53        3.83
price             12.25        0.71
n_images           1.63       -0.10
desc_len           3.58       -0.10
n_specs            1.73       -0.02
review_count      12.98        0.86


## 1d: Biến nhị phân

In [ ]:
fig, axs = panel_grid(len(bin_cols), ncol=min(len(bin_cols), 5), w=3.2, h=3.2)
rows2d = []
for ax, c in zip(axs, bin_cols):
    vc = df[c].map({0: "0", 1: "1"}).fillna("thiếu").value_counts()
    vc = vc.reindex([k for k in ["0", "1", "thiếu"] if k in vc.index])
    ax.bar(vc.index, vc.values, color=["#9ecae1", "#4C72B0", "#bdbdbd"][:len(vc)])
    for i, (k, v) in enumerate(vc.items()):
        ax.text(i, v, f"{v:,}\n{v / len(df) * 100:.1f}%", ha="center", va="bottom", fontsize=8)
        rows2d.append({"bien": c, "gia_tri": k, "n": int(v), "pct": v / len(df) * 100})
    ax.set_ylim(top=vc.max() * 1.25)
    ax.set_title(label(c), fontsize=9)
fig.suptitle("Bước 1d. Biến nhị phân (số sản phẩm và tỷ lệ)", y=1.03)
fig.tight_layout()
save(fig, "1d_nhi_phan")
tab2d = pd.DataFrame(rows2d)
print(tab2d.round(1).to_string(index=False))
tab2d.to_csv(OUT / "bang_1d_nhi_phan.csv", index=False, encoding="utf-8-sig")


=== Bước 1d. Biến nhị phân ===
  -> đã lưu 1d_nhi_phan.png
           bien gia_tri    n  pct
      has_video       0 1548 76.1
      has_video       1  486 23.9
  s_is_official       0 1079 53.0
  s_is_official       1  955 47.0
is_tiki_trading       0 1345 66.1
is_tiki_trading       1  689 33.9
     is_branded       0  337 16.6
     is_branded       1 1697 83.4
    in_velocity       0   38  1.9
    in_velocity       1 1996 98.1


## 1e: Biến phân loại

In [ ]:
for c in cat_cols:
    t = df.groupby(c).agg(n=("product_id", "size"), TV_quantity_sold=("quantity_sold", "median"),
                           TV_sold_log=("sold_log", "median")).sort_values("n", ascending=False)
    t["pct"] = t.n / len(df) * 100
    print(t.round(1).to_string())
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(t) + 1.5))
    ax.barh(t.index[::-1], t.n[::-1], color="#4C72B0")
    for i, (n, p) in enumerate(zip(t.n[::-1], t.pct[::-1])):
        ax.text(n, i, f" {n:,} ({p:.1f}%)", va="center", fontsize=8)
    ax.set_xlim(right=t.n.max() * 1.25)
    ax.set_title(f"Bước 1e. Số sản phẩm theo {c}")
    fig.tight_layout()
    save(fig, f"1e_{c}")
    t.to_csv(OUT / f"bang_1e_{c}.csv", encoding="utf-8-sig")


=== Bước 1e. Biến phân loại ===
                                    n  TV_quantity_sold  TV_sold_log   pct
main_category                                                             
Laptop - Máy Vi Tính - Linh kiện  393              23.0          3.2  19.3
Thiết Bị Số - Phụ Kiện Số         295              57.0          4.1  14.5
Nhà Sách Tiki                     209             613.0          6.4  10.3
Nhà Cửa - Đời Sống                196              77.5          4.4   9.6
Điện Gia Dụng                     196              39.5          3.7   9.6
Thể Thao - Dã Ngoại               192              24.0          3.2   9.4
Ô Tô - Xe Máy - Xe Đạp            185              13.0          2.6   9.1
Thời trang nam                    166              14.5          2.7   8.2
Đồ Chơi - Mẹ & Bé                 160             415.0          6.0   7.9
Điện Thoại - Máy Tính Bảng         42              42.0          3.8   2.1
  -> đã lưu 1e_main_category.png


In [10]:
PDF.close()
print(f" Mọi biểu đồ và bảng nằm trong thư mục: {OUT.resolve()}")

 Mọi biểu đồ và bảng nằm trong thư mục: E:\DA cơ bản\CK\N14 - Tiki\eda_donbien
